# Mean-Difference Vector Evaluation (Quick Demo)

This notebook loads the pre-generated on/off-policy dataset and the
pre-extracted mean-difference vector and runs a small evaluation (by
default on a single example).

In [15]:
import os
from pathlib import Path
TARGET = Path('data/on_policy_persona.json')
root = Path.cwd()
if not TARGET.exists():
    for candidate in [root, *root.parents]:
        candidate_path = candidate / 'data' / 'on_policy_persona.json'
        if candidate_path.exists():
            os.chdir(candidate)
            print(f'Changed working directory to {candidate}')
            break
else:
    print(f'Working directory: {Path.cwd()}')

Working directory: /Users


In [13]:
import os
from pathlib import Path

ROOT = Path('..').resolve()
os.chdir(ROOT)
print('Project root:', ROOT)

Project root: /Users


In [14]:
import sys
ROOT = Path('.')
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print('PYTHONPATH updated:', sys.path[-1])

PYTHONPATH updated: .


In [10]:
from policy_vector_pipeline import load_dataset, MeanDifferenceVector
from scripts.evaluate_vector import compute_stats, load_model
from policy_vector_pipeline import ActivationCollector
from transformers import logging
import torch

logging.set_verbosity_error()

ModuleNotFoundError: No module named 'policy_vector_pipeline'

In [13]:
dataset_path = Path('data/on_policy_persona.json')
vector_path = Path('artifacts/qwen3_onpolicy_mean.pt')
model_name = 'Qwen/Qwen3-4B'

dataset = load_dataset(dataset_path)
vector = MeanDifferenceVector.load(vector_path)
layer_ids = sorted(vector.layer_vectors.keys())
dataset.metadata.get('reasoning_model'), len(dataset.examples), layer_ids[:3]

('Qwen/Qwen3-4B', 60, [18, 19, 20])

In [14]:
model, tokenizer = load_model(model_name, device_map='auto', dtype='auto')
collector = ActivationCollector(
    model,
    tokenizer,
    layers=layer_ids,
    reduction='mean',
    response_only=True,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [7]:
activations = collector.collect_dataset(dataset, progress=True, limit=1)

KeyboardInterrupt: 

In [10]:
stats = compute_stats(activations, vector)
top = stats[0]
top_layer = top['layer']
top

{'layer': 34,
 'mean_on': -155.4801788330078,
 'mean_off': -167.4405517578125,
 'diff': 11.960372924804688,
 'std_on': 1.960364390969097,
 'std_off': 2.331931669578779,
 'cohens_d': 5.552184293839461,
 'threshold': -161.46036529541016,
 'on_acc': 1.0,
 'off_acc': 1.0,
 'overall_acc': 1.0}

In [11]:
proj_layer = top_layer
vec = vector.layer_vectors[proj_layer].to(torch.float32)
vec /= torch.linalg.norm(vec) + 1e-8
proj_on = torch.stack(activations['on'][proj_layer]).to(torch.float32) @ vec
proj_off = torch.stack(activations['off'][proj_layer]).to(torch.float32) @ vec
proj_on, proj_off

(tensor([-154.4665, -159.3103, -153.8077, -154.7269, -155.0895]),
 tensor([-167.1023, -171.2284, -168.0110, -166.9004, -163.9606]))